# 11 · Physics-Informed DeepONet — Operators Without Labelled Data

**Level:** Research
**Prerequisites:** `09_deeponet`, `07_inverse_parameter_estimation`

---

## 🧭 The Problem with Supervised DeepONet (NB-09)

In NB-09, we trained DeepONet on 1000 (input function, output function) labelled pairs. To generate those labels we ran numerical integration 1000 times — cheap for the antiderivative, but for real PDE problems:

- **Navier–Stokes at high Re**: each simulation takes hours on a cluster
- **Structural mechanics**: finite element runs take hours per loading condition
- **Plasma physics**: days per configuration

Generating 1000 labelled pairs becomes prohibitively expensive. Can we train the operator *without* any labelled output functions?

## 🔑 The Physics-Informed DeepONet Idea

Yes — by replacing the output labels with the **governing equation**.

**Key insight:** If $s(y) = G(u)(y)$ is the correct output for input $u$, then by definition:
$$\frac{d}{dy} G(u)(y) = u(y) \qquad (\text{for the antiderivative operator})$$

This is the **defining equation of the operator** itself. We can enforce it directly by:
1. Differentiating the DeepONet output $G(u)(y)$ with respect to the trunk input $y$ (via autograd)
2. Penalising the residual $\partial_y G(u)(y) - u(y)$

No output labels $s$ are needed — only the input functions $u$ and the equation they satisfy.

### From supervised to physics-informed: the swap

| Component | Supervised (NB-09) | Physics-Informed (NB-11) |
|---|---|---|
| Branch inputs | $u$ at sensors | $u$ at sensors (same) |
| Trunk inputs | Query $y$ | Query $y$ with `requires_grad=True` |
| Loss | `MSE(G(u)(y), s_true)` | `MSE(∂_y G(u)(y), u(y))` |
| Labels needed | Yes — 1000 exact outputs | **No** — only the equation |
| BC enforced | Via labels at $y=0$ | Explicit BC loss: `G(u)(0) = 0` |

### The chain rule through the trunk

The key technical step is differentiating $G(u)(y)$ with respect to $y$:

$$\frac{\partial}{\partial y} G(u)(y) = \frac{\partial}{\partial y}\left[\sum_{k=1}^p b_k(u) \cdot t_k(y)\right] = \sum_{k=1}^p b_k(u) \cdot t_k'(y)$$

where $t_k'(y) = \partial t_k / \partial y$ is computed by autograd through the trunk network. The branch does not change (it encodes $u$, which doesn't depend on $y$). Only the trunk is differentiated.

In PyTorch: set `y.requires_grad_(True)`, compute `s = model(u_branch, y)`, then:
```python
s_y = torch.autograd.grad(s, y, ones_like(s), create_graph=True)[0]
```

This gives $\partial s / \partial y = \partial G(u)(y) / \partial y$ — exactly what we need.

### What you will learn
1. Differentiating the DeepONet output w.r.t. the trunk (query) input — autograd through an operator
2. Building a physics loss for an *operator* (without any solution labels)
3. Enforcing boundary conditions as a separate loss term
4. Comparing PI-DeepONet vs. supervised DeepONet accuracy

### References
- Wang, Wang & Perdikaris (2021), *Learning the solution operator of parametric PDEs with physics-informed DeepONets*, Sci. Adv. 7:eabi8605
- Goswami et al. (2022), *Physics-informed deep neural operator networks* (variational form)
- Mishra & Molinaro (2023), *Estimates on the generalisation error of physics-informed neural operator*

## 1 · The problem, restated as physics

We learn the **antiderivative operator** $G: u \mapsto s$ defined by:

$$\frac{d s}{d y} = u(y), \qquad s(0) = 0$$

**Physics loss:** At random query points $y \in [0,1]$, enforce:
$$r(y) = \frac{\partial}{\partial y} G(u)(y) - u(y) = 0$$

**Boundary loss:** Enforce the initial condition:
$$G(u)(0) = 0 \quad \text{for all training input functions } u$$

**No ground-truth outputs anywhere** — the network learns the operator purely from the equation it must satisfy.

### Why two loss terms?

The ODE $ds/dy = u(y)$ has infinitely many solutions differing by an additive constant:
$$s(y) = \int_0^y u(\tau)d\tau + C, \qquad C \text{ arbitrary}$$

The physics loss alone does not pin down $C$. The boundary condition $s(0)=0$ selects the unique solution with $C=0$. Without both loss terms, the network could converge to a wrong constant-shifted version.

This is analogous to the standard PINN setup where the physics loss alone doesn't determine the solution — you also need ICs and BCs to make the problem well-posed.

### How the residual is computed through the operator

For each mini-batch:
1. Sample batch of input functions: $\{u_i\}_{i=1}^B$ (branch inputs, shape `(B, m)`)
2. Sample random query locations: $y \in [0,1]$ with `requires_grad=True` (trunk inputs)
3. Forward: `s = model(u_batch, y)` → shape `(B, 1)`
4. Differentiate: `s_y = autograd.grad(s, y, ...)` → $\partial s / \partial y$, shape `(B, 1)`
5. Compute residual: `r = s_y - u(y)` where `u(y)` = the input function evaluated at query $y$

Step 5 requires evaluating $u(y)$ at a continuous query location — since $u$ is only known at the $m$ sensor points, we use **linear interpolation** between the two nearest sensor values.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

m = 100
x_grid = np.linspace(0,1,m)

def random_function(n=1, ls=0.2):
    X=x_grid.reshape(-1,1)
    K=np.exp(-(X-X.T)**2/(2*ls**2))+1e-8*np.eye(m)
    L=np.linalg.cholesky(K)
    return (L@np.random.randn(m,n)).T

def antideriv(u):   # ONLY used to check accuracy at the end
    dx=x_grid[1]-x_grid[0]; s=np.zeros_like(u)
    s[:,1:]=np.cumsum((u[:,1:]+u[:,:-1])/2*dx,axis=1); return s


Using device: cpu


In [ ]:
class DeepONet(nn.Module):
    def __init__(self, m, p=50, width=100):
        super().__init__()
        self.branch=nn.Sequential(nn.Linear(m,width),nn.Tanh(),
                                  nn.Linear(width,width),nn.Tanh(),nn.Linear(width,p))
        self.trunk =nn.Sequential(nn.Linear(1,width),nn.Tanh(),
                                  nn.Linear(width,width),nn.Tanh(),nn.Linear(width,p),nn.Tanh())
        self.b0=nn.Parameter(torch.zeros(1))
    def forward(self,u,y):
        return torch.sum(self.branch(u)*self.trunk(y),dim=1,keepdim=True)+self.b0

model=DeepONet(m).to(device)


### 📌 Architecture — same as NB-09, used differently

The `DeepONet` class is identical to NB-09. The difference is entirely in **how we compute the loss** — specifically, we differentiate through the trunk input $y$ instead of comparing to labelled outputs.

The critical change during training:
```python
# NB-09 (supervised):
y = torch.rand(...)          # No requires_grad needed
s = model(u_batch, y)
loss = MSE(s, s_true)        # Compare to known output

# NB-11 (physics-informed):
y = torch.rand(..., requires_grad=True)   # ← MUST have requires_grad
s = model(u_batch, y)
s_y = autograd.grad(s, y, ...)            # ← differentiate w.r.t. query
loss = MSE(s_y, u_at_y) + MSE(model(u_bc, y_0), 0)   # physics + BC
```

The network architecture is unchanged — the physics is enforced purely through the loss construction. This is a powerful pattern: **same architecture, different learning signal**.

In [ ]:
# --- Training set: input functions + their sensor values interpolated at query y ---
N_fun = 1500
U = random_function(N_fun)                        # (N,m) input functions
U_t = torch.tensor(U, dtype=torch.float32, device=device)

def sample_batch(bs):
    idx = np.random.randint(0, N_fun, bs)
    u_b = U_t[idx]                                 # (bs, m)
    y   = torch.rand(bs,1,device=device, requires_grad=True)   # query points
    # u(y): linear-interp the input function at y (differentiable in y not needed for RHS)
    yi = (y.detach().cpu().numpy().ravel()*(m-1))
    lo = np.floor(yi).astype(int); lo=np.clip(lo,0,m-2); fr=yi-lo
    u_at_y = U[idx,lo]*(1-fr)+U[idx,lo+1]*fr
    u_at_y = torch.tensor(u_at_y, dtype=torch.float32, device=device).view(-1,1)
    return u_b, y, u_at_y

# boundary batch (y=0)
def boundary_batch(bs):
    idx=np.random.randint(0,N_fun,bs)
    return U_t[idx], torch.zeros(bs,1,device=device)


### 📌 The `sample_batch` function — why interpolation is needed

```python
def sample_batch(bs):
    idx  = randint(0, N_fun, bs)         # pick bs random input functions
    u_b  = U_t[idx]                      # branch input: discretised u at sensors

    y    = rand(bs, 1, requires_grad=True)   # random query locations in [0,1]

    # Compute u(y) by linear interpolation between sensor values:
    yi = y.detach().numpy() * (m-1)      # map y ∈ [0,1] to sensor index
    lo = floor(yi).astype(int)           # lower sensor index
    fr = yi - lo                         # fractional offset
    u_at_y = U[idx, lo] * (1-fr) + U[idx, lo+1] * fr   # linear interp
```

**Why is interpolation necessary?**

The physics loss requires $u(y)$ for arbitrary $y \in [0,1]$, but $u$ is only discretised at the $m=100$ fixed sensor locations $\{x_0, x_1, ..., x_{99}\}$. When $y$ falls between sensor locations, we approximate $u(y)$ by linear interpolation between the two nearest sensor values.

**Why not make `y` exactly a sensor location?**
If we only queried at sensor locations, the trunk net would never be forced to interpolate between them — it could learn a lookup table rather than a continuous function. Random query locations $y$ force the trunk to learn genuine continuity in $y$.

**Why `y.detach()` for the interpolation but keep `y` for autograd?**
The interpolation (a NumPy computation) runs outside PyTorch's autograd. We only need `y` with gradients for the `autograd.grad(s, y, ...)` call inside the training loop. Using `y.detach()` for the interpolation avoids accidentally breaking the computation graph.

**The boundary batch:**
```python
def boundary_batch(bs):
    return U_t[rand_indices], zeros(bs, 1)   # y=0 for all functions
```
All input functions must satisfy $G(u)(0) = 0$. We evaluate the operator at $y=0$ for a fresh random batch of input functions each iteration, forcing the BC to hold across the whole function family — not just for one fixed $u$.

In [ ]:
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
EPOCHS=10000
for ep in range(EPOCHS):
    opt.zero_grad()
    u_b,y,u_at_y = sample_batch(2000)
    s = model(u_b,y)
    s_y = torch.autograd.grad(s,y,torch.ones_like(s),create_graph=True)[0]
    L_phys = torch.mean((s_y - u_at_y)**2)         # ds/dy = u(y)
    ub,y0 = boundary_batch(500)
    L_bc = torch.mean(model(ub,y0)**2)             # s(0)=0
    loss = L_phys + L_bc
    loss.backward(); opt.step()
    if ep%2000==0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | phys {L_phys.item():.2e} bc {L_bc.item():.2e}")
print("done — trained with ZERO solution labels")


### 📌 Training loop — the physics-informed operator loss in detail

```python
u_b, y, u_at_y = sample_batch(2000)        # 2000 (function, query, u(y)) triples

s   = model(u_b, y)                         # shape: (2000, 1) — operator output
s_y = autograd.grad(s, y, ones_like(s), create_graph=True)[0]   # ∂G(u)/∂y

L_phys = mean((s_y - u_at_y)**2)           # should equal zero: ∂s/∂y = u(y)
L_bc   = mean(model(ub, y0)**2)            # should equal zero: s(0) = 0
```

**Understanding the physics loss term:**

`s_y - u_at_y` should be zero everywhere if $G$ is correct:
- `s_y = ∂G(u)(y)/∂y`: the derivative of the operator output w.r.t. the query location
- `u_at_y = u(y)`: the input function evaluated at the query location

The antiderivative by definition satisfies $\frac{d}{dy}\int_0^y u(\tau)d\tau = u(y)$. So we're literally enforcing the Fundamental Theorem of Calculus as a neural network training objective.

**How backpropagation works through the differentiation:**

The computation graph is:
```
y (leaf, requires_grad=True)
  → trunk(y)              [through trunk network layers]
  → b·t + bias            [dot product with branch output]
  → s                     [operator output]
  → s_y = ∂s/∂y          [via autograd, kept in graph by create_graph=True]
  → loss = (s_y - u)²    [physics residual]
  → loss.backward()       [updates ALL parameters: branch, trunk, bias]
```

The trunk receives gradient signal *from two paths*: directly from `s` (via the data loss in NB-09) or indirectly through `s_y` (the physics loss here). The branch receives gradient only through the dot product.

**Why `create_graph=True` in the grad call?**

Without it, `s_y` is a "dead" tensor — it has a value but no graph. `loss.backward()` would not be able to differentiate through `s_y` to update the model parameters. `create_graph=True` keeps the computational graph alive through the differentiation step, enabling second-order differentiation.

In [ ]:
# --- Test against the true antiderivative (which the model never saw) ---
U_test=random_function(3); S_test=antideriv(U_test)
y_full=torch.tensor(x_grid,dtype=torch.float32,device=device).view(-1,1)
fig,ax=plt.subplots(1,3,figsize=(15,4))
for i in range(3):
    br=torch.tensor(np.repeat(U_test[i:i+1],m,axis=0),dtype=torch.float32,device=device)
    with torch.no_grad():
        pred=model(br,y_full).cpu().numpy().ravel()
    err=np.linalg.norm(pred-S_test[i])/np.linalg.norm(S_test[i])
    ax[i].plot(x_grid,S_test[i],'k-',lw=2,label='true ∫u')
    ax[i].plot(x_grid,pred,'r--',lw=2,label='PI-DeepONet')
    ax[i].plot(x_grid,U_test[i],color='gray',alpha=0.5,label='input u')
    ax[i].set_title(f'unseen #{i+1} (L2={err:.2e})'); ax[i].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 2 · Why this matters — the physics-informed operator learning paradigm

### Reading the test results

The test plots show the PI-DeepONet applied to **3 unseen input functions** that were never in the training set (not even their antiderivatives). The L2 errors confirm whether the operator has genuinely learned the integration rule, not memorised training pairs.

Typical observations:
- **PI-DeepONet accuracy** (L2 ~ `1e-2` to `1e-3`): slightly worse than supervised DeepONet for the same number of epochs — labels are a stronger training signal
- **Generalization quality**: both models generalise well because the physics constraint guides the operator toward the correct functional form

### The data-free training advantage

Combining operator learning with physics gives the best of both worlds:

| Capability | Classical solver | Supervised DeepONet | PI-DeepONet |
|---|---|---|---|
| Requires labelled outputs? | N/A | **Yes** (expensive) | **No** |
| Requires governing equation? | Yes | No (data-driven) | **Yes** |
| Inference speed | Slow | Fast | Fast |
| Label generation cost | N/A | High | **Zero** |
| Works when PDE is known? | Yes | Yes (with labels) | **Yes (without labels)** |

**The sweet spot:** PI-DeepONet is ideal when:
1. The PDE is known (first-principles physics is available)
2. The solution is hard/expensive to simulate
3. You need fast inference for many input conditions

### Scaling to time-dependent PDEs

The antiderivative is a 1D operator for illustration. The same recipe extends directly to time-dependent PDEs by including $t$ in the trunk coordinates:

$$G(u_0)(x, t) = s(x, t), \qquad s_t = \alpha s_{xx}, \quad s(x, 0) = u_0(x)$$

The trunk encodes $(x, t)$; the physics loss enforces $\partial_t s = \alpha \partial_{xx} s$ at random collocation $(x, t)$ pairs; the IC loss enforces $s(x, 0) = u_0(x)$. No labelled $s(x, t)$ needed — only the heat equation and the IC family.

This is exactly what Wang, Wang & Perdikaris (2021) demonstrated for the Burgers equation and the reaction-diffusion system — both much harder problems than integration, solved without a single labelled output.